# NLP Lab Assignment 4 — N-gram Language Models

This notebook uses the Telugu sentence data created in Assignment 1.

Requirements:
- Select exactly 1,000,000 valid sentences
- Training: 998,000 sentences
- Development: 1,000 sentences
- Test: 1,000 sentences
- Build Unigram, Bigram, Trigram and Quadrigram language models
- Apply Add-One/Laplace smoothing
- Evaluate using perplexity


## 1. Imports


In [13]:
import re
import math
import random
import time
from collections import Counter

import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Configuration


In [14]:
INPUT_FILE = "../ASS-1/all_sentences.parquet"

TRAIN_FILE = "train.parquet"
DEV_FILE = "dev.parquet"
TEST_FILE = "test.parquet"

TOTAL_SENTENCES = 1_000_000
TRAIN_SIZE = 998_000
DEV_SIZE = 1_000
TEST_SIZE = 1_000

# Read extra candidates because a small number of rows are noise-only.
CANDIDATE_SENTENCES = 1_200_000

SEED = 42
random.seed(SEED)

assert TRAIN_SIZE + DEV_SIZE + TEST_SIZE == TOTAL_SENTENCES

print("Configuration loaded.")


Configuration loaded.


## 3. Assignment-1 word tokenizer


In [15]:
TOKEN_PATTERN = re.compile(r"""
    https?://[^\s]+|                       # URLs
    \b[\w.+-]+@[\w.-]+\.\w+\b|             # Email IDs
    \d{1,2}[/-]\d{1,2}[/-]\d{2,4}|         # Dates
    \d+\.\d+|                              # Decimal numbers
    [\u0C66-\u0C6F]+|                      # Telugu digits
    \d+|                                   # Integers
    [\u0C00-\u0C7F]+|                     # Telugu words
    [A-Za-z]+|                            # English words
    \.{3}|                                # Ellipsis
    [.,!?;:()"'\-]                        # Punctuation
""", re.VERBOSE)

def tokenize(sentence):
    return TOKEN_PATTERN.findall(sentence)

print("Tokenizer ready.")


Tokenizer ready.


## 4. Check Assignment-1 sentence dataset


In [16]:
pf = pq.ParquetFile(INPUT_FILE)

print("Total sentences:", f"{pf.metadata.num_rows:,}")
print("Row groups:", pf.num_row_groups)
print("Schema:")
print(pf.schema)


Total sentences: 77,685,001
Row groups: 777
Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 text (String);
}



## 5. Select enough random row groups


In [17]:
row_group_ids = list(range(pf.num_row_groups))
random.shuffle(row_group_ids)

selected_groups = []
available_rows = 0

for rg in row_group_ids:
    selected_groups.append(rg)
    available_rows += pf.metadata.row_group(rg).num_rows

    if available_rows >= CANDIDATE_SENTENCES:
        break

print("Selected row groups:", len(selected_groups))
print("Candidate rows available:", f"{available_rows:,}")


Selected row groups: 12
Candidate rows available: 1,200,000


## 6. Load candidate sentences


In [18]:
table = pf.read_row_groups(selected_groups, columns=["text"])
candidate_sentences = table["text"].to_pylist()

print("Loaded candidate sentences:", f"{len(candidate_sentences):,}")


Loaded candidate sentences: 1,200,000


## 7. Remove noise-only sentences

Rows such as `[…]`, `*`, `●`, zero-width characters and replacement
characters produce no tokens with the Assignment-1 tokenizer, so they are excluded.


In [19]:
valid_sentences = []
invalid_sentences = []

for sentence in candidate_sentences:
    if sentence is None:
        continue

    sentence = sentence.strip()

    if not sentence:
        continue

    tokens = tokenize(sentence)

    if tokens:
        valid_sentences.append(sentence)
    else:
        invalid_sentences.append(sentence)

print("Candidate:", f"{len(candidate_sentences):,}")
print("Valid:", f"{len(valid_sentences):,}")
print("Invalid/noise:", f"{len(invalid_sentences):,}")


Candidate: 1,200,000
Valid: 1,199,850
Invalid/noise: 150


In [20]:
if len(valid_sentences) < TOTAL_SENTENCES:
    raise ValueError(
        f"Only {len(valid_sentences):,} valid sentences found. "
        f"Increase CANDIDATE_SENTENCES."
    )

print("Enough valid sentences are available.")


Enough valid sentences are available.


## 8. Select exactly 1,000,000 valid sentences


In [21]:
random.shuffle(valid_sentences)
sentences = valid_sentences[:TOTAL_SENTENCES]

print("Final sentences:", f"{len(sentences):,}")
assert len(sentences) == TOTAL_SENTENCES


Final sentences: 1,000,000


## 9. Split into train/dev/test


In [22]:
train_sentences = sentences[:TRAIN_SIZE]
dev_sentences = sentences[TRAIN_SIZE:TRAIN_SIZE + DEV_SIZE]
test_sentences = sentences[TRAIN_SIZE + DEV_SIZE:]

print("Training:", f"{len(train_sentences):,}")
print("Development:", f"{len(dev_sentences):,}")
print("Test:", f"{len(test_sentences):,}")

assert len(train_sentences) == 998_000
assert len(dev_sentences) == 1_000
assert len(test_sentences) == 1_000


Training: 998,000
Development: 1,000
Test: 1,000


## 10. Save the corrected split


In [23]:
def save_sentence_file(sentences, filename):
    pq.write_table(
        pa.table({"text": sentences}),
        filename,
        compression="snappy"
    )
    print("Saved:", filename)

save_sentence_file(train_sentences, TRAIN_FILE)
save_sentence_file(dev_sentences, DEV_FILE)
save_sentence_file(test_sentences, TEST_FILE)


Saved: train.parquet
Saved: dev.parquet
Saved: test.parquet


## 11. Prepare sentences with language-model boundary tokens


In [24]:
def prepare_sentence(sentence):
    return ["<START>"] + tokenize(sentence) + ["<END>"]

for sentence in train_sentences[:5]:
    print("Sentence:", sentence)
    print("Tokens:", prepare_sentence(sentence))
    print()


Sentence: దీనిపై పాట్నాలో ఆయన మాట్లాడుతూ, మోదీ నుంచి దేశం చాలా ఆశించిందని అన్నారు.
Tokens: ['<START>', 'దీనిపై', 'పాట్నాలో', 'ఆయన', 'మాట్లాడుతూ', ',', 'మోదీ', 'నుంచి', 'దేశం', 'చాలా', 'ఆశించిందని', 'అన్నారు', '.', '<END>']

Sentence: రాయుడి రిటైర్మెంట్‌పై యోగ్‌‌రాజ్‌ మాట్లాడుతూ
Tokens: ['<START>', 'రాయుడి', 'రిటైర్మెంట్', 'పై', 'యోగ్', 'రాజ్', 'మాట్లాడుతూ', '<END>']

Sentence: 60యేళ్ల మాయాబజార్‌పై అటు పత్రికల్లోను, ఇటు చానెల్స్‌లోనూ కథనాలు చూస్తుంటే -అలాంటి చిత్రాలు మళ్లీ మన తెలుగు చిత్రసీమలో రావా?
Tokens: ['<START>', '60', 'యేళ్ల', 'మాయాబజార్', 'పై', 'అటు', 'పత్రికల్లోను', ',', 'ఇటు', 'చానెల్స్', 'లోనూ', 'కథనాలు', 'చూస్తుంటే', '-', 'అలాంటి', 'చిత్రాలు', 'మళ్లీ', 'మన', 'తెలుగు', 'చిత్రసీమలో', 'రావా', '?', '<END>']

Sentence: మనసులో సీఎం అవ్వాలనే ఆలోచన ఉండటం వల్లే ఆయన గట్టిగా మాట్లాడలేదనే వాదన వినిపిస్తోంది పొలిటికల్ సర్కిల్‌లో.
Tokens: ['<START>', 'మనసులో', 'సీఎం', 'అవ్వాలనే', 'ఆలోచన', 'ఉండటం', 'వల్లే', 'ఆయన', 'గట్టిగా', 'మాట్లాడలేదనే', 'వాదన', 'వినిపిస్తోంది', 'పొలిటికల్', 'సర్కిల్', 

## 12. Build vocabulary from TRAINING DATA ONLY


In [25]:
vocab = {"<START>", "<END>", "<UNK>"}

for sentence in train_sentences:
    vocab.update(tokenize(sentence))

V = len(vocab)

print("Vocabulary size V:", f"{V:,}")


Vocabulary size V: 593,723


## 13. Assign integer IDs


In [26]:
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for token, idx in token_to_id.items()}

START_ID = token_to_id["<START>"]
END_ID = token_to_id["<END>"]
UNK_ID = token_to_id["<UNK>"]

print("Vocabulary size:", f"{len(token_to_id):,}")
print("START ID:", START_ID)
print("END ID:", END_ID)
print("UNK ID:", UNK_ID)


Vocabulary size: 593,723
START ID: 407555
END ID: 429217
UNK ID: 555329


## 14. Verify token-to-ID conversion


In [27]:
sample_sentence = train_sentences[0]
sample_tokens = prepare_sentence(sample_sentence)
sample_ids = [token_to_id.get(t, UNK_ID) for t in sample_tokens]

print("Sentence:")
print(sample_sentence)

print("\nTokens:")
print(sample_tokens)

print("\nIDs:")
print(sample_ids)


Sentence:
దీనిపై పాట్నాలో ఆయన మాట్లాడుతూ, మోదీ నుంచి దేశం చాలా ఆశించిందని అన్నారు.

Tokens:
['<START>', 'దీనిపై', 'పాట్నాలో', 'ఆయన', 'మాట్లాడుతూ', ',', 'మోదీ', 'నుంచి', 'దేశం', 'చాలా', 'ఆశించిందని', 'అన్నారు', '.', '<END>']

IDs:
[407555, 312586, 284802, 474961, 188044, 447734, 551487, 411947, 334135, 518513, 89506, 13834, 524463, 429217]


## 15. Create n-gram counters


In [28]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()
quadrigram_counts = Counter()

print("N-gram counters created.")


N-gram counters created.


## 16. Build Unigram, Bigram, Trigram and Quadrigram models

Only n-grams that actually occur in the training data are stored.
Unseen n-grams are handled later by Laplace smoothing.


In [29]:
start_time = time.time()
processed = 0

for sentence in train_sentences:
    ids = [
        token_to_id.get(token, UNK_ID)
        for token in prepare_sentence(sentence)
    ]

    unigram_counts.update(ids)
    bigram_counts.update(zip(ids, ids[1:]))
    trigram_counts.update(zip(ids, ids[1:], ids[2:]))
    quadrigram_counts.update(
        zip(ids, ids[1:], ids[2:], ids[3:])
    )

    processed += 1

    if processed % 100_000 == 0 or processed == TRAIN_SIZE:
        print(f"Processed {processed:,} sentences")

elapsed = time.time() - start_time

print("\nN-gram counting completed.")
print("Final processed:", f"{processed:,}")
print("Time:", f"{elapsed:.2f} seconds")

assert processed == TRAIN_SIZE


Processed 100,000 sentences
Processed 200,000 sentences
Processed 300,000 sentences
Processed 400,000 sentences
Processed 500,000 sentences
Processed 600,000 sentences
Processed 700,000 sentences
Processed 800,000 sentences
Processed 900,000 sentences
Processed 998,000 sentences

N-gram counting completed.
Final processed: 998,000
Time: 50.61 seconds


## 17. N-gram statistics


In [30]:
total_tokens = sum(unigram_counts.values())

print("Unique unigrams:", f"{len(unigram_counts):,}")
print("Unique bigrams:", f"{len(bigram_counts):,}")
print("Unique trigrams:", f"{len(trigram_counts):,}")
print("Unique quadrigrams:", f"{len(quadrigram_counts):,}")
print("Total training tokens:", f"{total_tokens:,}")


Unique unigrams: 593,722
Unique bigrams: 4,813,387
Unique trigrams: 8,244,124
Unique quadrigrams: 9,197,704
Total training tokens: 13,586,573


## 18. Most frequent unigrams


In [31]:
for token_id, count in unigram_counts.most_common(20):
    print(f"{id_to_token[token_id]:20s} -> {count:,}")


.                    -> 1,055,742
<START>              -> 998,000
<END>                -> 998,000
,                    -> 391,484
ఈ                    -> 119,457
లో                   -> 75,595
కూడా                 -> 53,804
ఆ                    -> 38,571
-                    -> 37,446
:                    -> 34,321
కు                   -> 33,741
ఒక                   -> 32,878
)                    -> 32,208
నుంచి                -> 32,200
(                    -> 31,680
ను                   -> 31,654
ఉంది                 -> 31,114
మరియు                -> 26,508
నుండి                -> 26,344
కి                   -> 26,339


## 19. Add-One / Laplace Smoothing

The assignment specifies:

P(w_n | w_1,...,w_(n-1)) =
(C(w_1,...,w_n) + 1) / (C(w_1,...,w_(n-1)) + V)

Therefore:
- Unigram: (C(w) + 1) / (N + V)
- Bigram: (C(w1,w2) + 1) / (C(w1) + V)
- Trigram: (C(w1,w2,w3) + 1) / (C(w1,w2) + V)
- Quadrigram: (C(w1,w2,w3,w4) + 1) / (C(w1,w2,w3) + V)


In [32]:
def unigram_probability(w):
    return (unigram_counts[w] + 1) / (total_tokens + V)

def bigram_probability(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (
        unigram_counts[w1] + V
    )

def trigram_probability(w1, w2, w3):
    return (trigram_counts[(w1, w2, w3)] + 1) / (
        bigram_counts[(w1, w2)] + V
    )

def quadrigram_probability(w1, w2, w3, w4):
    return (quadrigram_counts[(w1, w2, w3, w4)] + 1) / (
        trigram_counts[(w1, w2, w3)] + V
    )

print("Laplace probability functions created.")


Laplace probability functions created.


## 20. Verify smoothing


In [33]:
print("Unigram example:", unigram_probability(UNK_ID))
print("Bigram example:", bigram_probability(START_ID, UNK_ID))
print("Trigram example:", trigram_probability(START_ID, UNK_ID, END_ID))
print(
    "Quadrigram example:",
    quadrigram_probability(START_ID, UNK_ID, UNK_ID, END_ID)
)


Unigram example: 7.052038970131512e-08
Bigram example: 6.28250015863313e-07
Trigram example: 1.6842871170562704e-06
Quadrigram example: 1.6842871170562704e-06


## 21. Convert evaluation sentences to IDs


In [34]:
def sentence_to_ids(sentence):
    tokens = prepare_sentence(sentence)
    return [token_to_id.get(token, UNK_ID) for token in tokens]

print("Development sentence:")
print(dev_sentences[0])

print("\nIDs:")
print(sentence_to_ids(dev_sentences[0]))


Development sentence:
గ్రామంలో అంగన్ వాడీ కేంద్రం, ఇతర పోషకాహార కేంద్రాలు, ఆశా కార్యకర్త ఉన్నాయి.

IDs:
[407555, 492189, 359922, 522326, 539347, 447734, 457896, 64135, 37584, 447734, 201347, 330871, 70191, 524463, 429217]


## 22. Sentence log probabilities


In [35]:
def unigram_log_probability(sentence):
    ids = sentence_to_ids(sentence)
    return sum(math.log(unigram_probability(w)) for w in ids)

def bigram_log_probability(sentence):
    ids = sentence_to_ids(sentence)
    return sum(
        math.log(bigram_probability(ids[i-1], ids[i]))
        for i in range(1, len(ids))
    )

def trigram_log_probability(sentence):
    ids = sentence_to_ids(sentence)
    return sum(
        math.log(trigram_probability(ids[i-2], ids[i-1], ids[i]))
        for i in range(2, len(ids))
    )

def quadrigram_log_probability(sentence):
    ids = sentence_to_ids(sentence)
    return sum(
        math.log(
            quadrigram_probability(
                ids[i-3], ids[i-2], ids[i-1], ids[i]
            )
        )
        for i in range(3, len(ids))
    )

print("Log-probability functions created.")


Log-probability functions created.


## 23. Perplexity

PP = exp(-(1/N) * sum log P(w_i | context))

Lower perplexity indicates better prediction of the evaluation text.


In [36]:
def calculate_perplexity(sentences, model):
    total_log_probability = 0.0
    total_predictions = 0

    for sentence in sentences:
        ids = sentence_to_ids(sentence)

        if model == "unigram":
            for w in ids:
                total_log_probability += math.log(
                    unigram_probability(w)
                )
                total_predictions += 1

        elif model == "bigram":
            for i in range(1, len(ids)):
                total_log_probability += math.log(
                    bigram_probability(ids[i-1], ids[i])
                )
                total_predictions += 1

        elif model == "trigram":
            for i in range(2, len(ids)):
                total_log_probability += math.log(
                    trigram_probability(
                        ids[i-2], ids[i-1], ids[i]
                    )
                )
                total_predictions += 1

        elif model == "quadrigram":
            for i in range(3, len(ids)):
                total_log_probability += math.log(
                    quadrigram_probability(
                        ids[i-3], ids[i-2], ids[i-1], ids[i]
                    )
                )
                total_predictions += 1

        else:
            raise ValueError(
                "model must be unigram, bigram, trigram, or quadrigram"
            )

    cross_entropy = -total_log_probability / total_predictions
    return math.exp(cross_entropy)


## 24. Development-set evaluation


In [37]:
print("Calculating development-set perplexity...")

dev_unigram_pp = calculate_perplexity(dev_sentences, "unigram")
print("Unigram:", dev_unigram_pp)

dev_bigram_pp = calculate_perplexity(dev_sentences, "bigram")
print("Bigram:", dev_bigram_pp)

dev_trigram_pp = calculate_perplexity(dev_sentences, "trigram")
print("Trigram:", dev_trigram_pp)

dev_quadrigram_pp = calculate_perplexity(dev_sentences, "quadrigram")
print("Quadrigram:", dev_quadrigram_pp)


Calculating development-set perplexity...
Unigram: 3609.644515669766
Bigram: 23251.02456441066
Trigram: 172773.84105425948
Quadrigram: 357434.59950659703


## 25. Test-set evaluation


In [38]:
print("Calculating test-set perplexity...")

test_unigram_pp = calculate_perplexity(test_sentences, "unigram")
print("Unigram:", test_unigram_pp)

test_bigram_pp = calculate_perplexity(test_sentences, "bigram")
print("Bigram:", test_bigram_pp)

test_trigram_pp = calculate_perplexity(test_sentences, "trigram")
print("Trigram:", test_trigram_pp)

test_quadrigram_pp = calculate_perplexity(test_sentences, "quadrigram")
print("Quadrigram:", test_quadrigram_pp)


Calculating test-set perplexity...
Unigram: 3953.639757984167
Bigram: 25837.04395457594
Trigram: 187497.99341590787
Quadrigram: 380520.5529419914


## 26. Final comparison


In [39]:
results_df = pd.DataFrame({
    "Model": ["Unigram", "Bigram", "Trigram", "Quadrigram"],
    "Development Perplexity": [
        dev_unigram_pp,
        dev_bigram_pp,
        dev_trigram_pp,
        dev_quadrigram_pp
    ],
    "Test Perplexity": [
        test_unigram_pp,
        test_bigram_pp,
        test_trigram_pp,
        test_quadrigram_pp
    ]
})

results_df


,Model,Development Perplexity,Test Perplexity
0,Unigram,3609.644516,3953.639758
1,Bigram,23251.024564,25837.043955
2,Trigram,172773.841054,187497.993416
3,Quadrigram,357434.599507,380520.552942


In [40]:
print(
    f"{'Model':<15}"
    f"{'Development PP':>20}"
    f"{'Test PP':>20}"
)
print("-" * 55)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']:<15}"
        f"{row['Development Perplexity']:>20.4f}"
        f"{row['Test Perplexity']:>20.4f}"
    )


Model                Development PP             Test PP
-------------------------------------------------------
Unigram                   3609.6445           3953.6398
Bigram                   23251.0246          25837.0440
Trigram                 172773.8411         187497.9934
Quadrigram              357434.5995         380520.5529


## 27. Save final results


In [41]:
results_df.to_csv("ngram_results.csv", index=False)
print("Saved: ngram_results.csv")


Saved: ngram_results.csv


## 28. Final observations

- The language data comes from Assignment 1.
- Exactly 1,000,000 valid sentences are used.
- 998,000 sentences are used for training.
- 1,000 sentences are used for development.
- 1,000 sentences are used for testing.
- Unigram uses no previous-word context.
- Bigram uses one previous word.
- Trigram uses two previous words.
- Quadrigram uses three previous words.
- Add-One/Laplace smoothing gives non-zero probability to unseen n-grams.
- Perplexity is used for evaluation; lower perplexity indicates better performance.
